
# VK Practice — Отчёт по графикам и метрикам

Этот ноутбук воспроизводит шаги визуализации и проверки модели:
1) загрузка **модели** и **CV-метрик**;  
2) построение **графиков** (распределение полов, активность по времени суток, ROC/PR и пр.);  
3) расчёт **метрик на учебной подвыборке** (подмножество `train` с реальными метками).



In [ ]:

# --- Конфигурация ---
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Параметры
N_USERS = 10000        # сколько пользователей брать из train для учебной подвыборки
SEED = 42              # сид для воспроизводимости выбора пользователей
LIGHT = True           # True = быстрые признаки, False = расширенные (дольше)

# Пути (относительно корня проекта)
ROOT = Path('.').resolve()
DATA_DIR = ROOT / 'data'
MODELS_DIR = ROOT / 'models'
REPORTS_DIR = ROOT / 'reports'
GRAPHICS_DIR = ROOT / 'graphics' / 'figures_nb'
GRAPHICS_DIR.mkdir(parents=True, exist_ok=True)

# Добавим src в sys.path
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print('ROOT =', ROOT)
print('DATA_DIR =', DATA_DIR)
print('MODELS_DIR =', MODELS_DIR)
print('REPORTS_DIR =', REPORTS_DIR)
print('GRAPHICS_DIR =', GRAPHICS_DIR)


In [ ]:

# --- Импорты проектных модулей ---
try:
    from data_prep import read_csv_auto, build_event_level
    from features import agg_user_features
    print('Импортированы: data_prep.read_csv_auto, data_prep.build_event_level, features.agg_user_features')
except Exception as e:
    print('Не удалось импортировать из src. Убедись, что запускаешь ноутбук из корня проекта. Ошибка:', e)
    raise


In [ ]:

# --- Загрузка обученной модели и CV-метрик ---
import joblib

bundle_path = MODELS_DIR / 'best_model.joblib'
cv_path = REPORTS_DIR / 'cv_metrics.json'

bundle = joblib.load(bundle_path)
model = bundle['model']
model_threshold = float(bundle.get('threshold', 0.5))
feature_cols = list(bundle['feature_cols'])
model_name = bundle.get('model_name', 'unknown')

print(f'Model loaded: {model_name}')
print(f'- threshold: {model_threshold:.3f}')
print(f'- n_features: {len(feature_cols)}')

with open(cv_path, 'r', encoding='utf-8') as f:
    cv = json.load(f)

print('\nCV-метрики (OOF):')
for k in ['auc', 'f1', 'acc']:
    mean = cv[k]; std = cv[f'{k}_std']
    print(f'  {k.upper()}: {mean:.4f} ± {std:.4f}')
print('  threshold (из CV):', cv.get('threshold', model_threshold))


## Распределение полов (train_labels)

In [ ]:

labels = read_csv_auto(DATA_DIR / 'train_labels.csv')
counts = labels['target'].value_counts().sort_index()
plt.figure(figsize=(6,4))
plt.bar([f'Класс {i}' for i in counts.index], counts.values)
plt.title('Распределение полов (train)')
plt.ylabel('Число пользователей')
for i, v in enumerate(counts.values):
    plt.text(i, v, str(v), ha='center', va='bottom')
plt.tight_layout()
plt.savefig(GRAPHICS_DIR / '01_gender_distribution_nb.png', dpi=150)
plt.show()


## Учебная подвыборка и Event-level признаки

In [ ]:

# Загружаем события и справочники
train = read_csv_auto(DATA_DIR / 'train.csv')
rv = read_csv_auto(DATA_DIR / 'referer_vectors.csv')
geo = read_csv_auto(DATA_DIR / 'geo_info.csv')

# Выбираем пользователей из меток
n_total = labels['user_id'].nunique()
n_take = min(N_USERS, n_total)
users_subset = labels['user_id'].drop_duplicates().sample(n=n_take, random_state=SEED)

# Срез событий по этим пользователям
train_sub = train[train['user_id'].isin(users_subset)]
print(f'Users in subset: {n_take:,} | events: {len(train_sub):,}')

# Строим event-level (join справочников, извлекаем time/UA-поля)
train_ev = build_event_level(train_sub, rv, geo)
print('event-level shape:', train_ev.shape)
train_ev.head(3)


## Активность по времени суток (доли событий)

In [ ]:

# Используем hour, который рассчитывается в build_event_level
hour = train_ev['hour'].dropna().astype(int).clip(0, 23)
bins = [0,6,12,18,24]
labels_bins = ['Ночь','Утро','День','Вечер']
hist, _ = np.histogram(hour, bins=bins)
shares = hist / hist.sum()

plt.figure(figsize=(6,4))
plt.bar(labels_bins, shares)
plt.title('Активность по времени суток')
plt.ylabel('Доля событий')
for i, v in enumerate(shares):
    plt.text(i, v, f'{v*100:.1f}%', ha='center', va='bottom')
plt.ylim(0, max(shares)*1.15)
plt.tight_layout()
plt.savefig(GRAPHICS_DIR / '03_activity_part_of_day_nb.png', dpi=150)
plt.show()


## User-level признаки, метрики на учебной подвыборке

In [ ]:

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve, precision_recall_curve, confusion_matrix

# Собираем фичи
train_user = agg_user_features(train_ev, light=LIGHT).merge(labels, on='user_id', how='inner')

# Выравниваем признаки под модель
missing = [c for c in feature_cols if c not in train_user.columns]
for c in missing:
    train_user[c] = np.nan

X = train_user[feature_cols]
y_true = train_user['target'].astype(int).values

# Предсказания
proba = model.predict_proba(X)[:, 1]
y_pred = (proba >= model_threshold).astype(int)

# Метрики
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
auc = roc_auc_score(y_true, proba)

print(f'Учебная подвыборка — N={len(y_true)}')
print(f'  Accuracy: {acc:.4f}')
print(f'  F1:       {f1:.4f}')
print(f'  AUC:      {auc:.4f}')

# Сохраним
with open(GRAPHICS_DIR / 'metrics_nb.txt', 'w', encoding='utf-8') as f:
    f.write(f'ACC={acc:.4f}\nF1={f1:.4f}\nAUC={auc:.4f}\nTHR={model_threshold:.3f}\n')


## ROC-кривая (учебная подвыборка)

In [ ]:

fpr, tpr, _ = roc_curve(y_true, proba)
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label='ROC')
plt.plot([0,1],[0,1], linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC-кривая (учебная подвыборка)')
plt.legend()
plt.tight_layout()
plt.savefig(GRAPHICS_DIR / '04_roc_curve_nb.png', dpi=150)
plt.show()


## Precision–Recall кривая (учебная подвыборка)

In [ ]:

precision, recall, _ = precision_recall_curve(y_true, proba)
plt.figure(figsize=(6,5))
plt.plot(recall, precision)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision–Recall (учебная подвыборка)')
plt.tight_layout()
plt.savefig(GRAPHICS_DIR / '05_pr_curve_nb.png', dpi=150)
plt.show()


## Матрица ошибок (по выбранному порогу)

In [ ]:

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5,5))
plt.imshow(cm, interpolation='nearest')
plt.title('Матрица ошибок')
plt.colorbar()
ticks = [0,1]
plt.xticks(ticks, ['Предсказано 0','Предсказано 1'], rotation=45, ha='right')
plt.yticks(ticks, ['Истинно 0','Истинно 1'])
thresh = cm.max() / 2.0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, int(cm[i, j]), ha='center', va='center', color='white' if cm[i,j]>thresh else 'black')
plt.tight_layout()
plt.savefig(GRAPHICS_DIR / '06_confusion_matrix_nb.png', dpi=150)
plt.show()


## Распределение вероятностей по классам (учебная подвыборка)

In [ ]:

p0 = proba[y_true==0]
p1 = proba[y_true==1]
plt.figure(figsize=(6,4))
plt.hist(p0, bins=50, alpha=0.6, label='Класс 0')
plt.hist(p1, bins=50, alpha=0.6, label='Класс 1')
plt.xlabel('Предсказанная вероятность класса 1')
plt.ylabel('Частота')
plt.title('Распределение вероятностей по классам')
plt.legend()
plt.tight_layout()
plt.savefig(GRAPHICS_DIR / '07_proba_hist_nb.png', dpi=150)
plt.show()
